# Portable LSTM Filter Selector Lab

Objective: kiểm tra lại pipeline `base_prediction -> filter_probability -> selector` bằng các module `.py` hiện có trong repo, không copy logic thủ công vào notebook.

Success criteria:
- Load được artifact filter hiện tại.
- Gọi trực tiếp `src.models.selection.filter_signal`.
- Refit selector chỉ trên split `train`, apply sang `val`.
- So kết quả notebook với CSV report đã tạo từ script full run.


## Setup

Notebook này giả định bạn mở từ repo root hoặc từ thư mục `notebooks/`. Cell dưới sẽ tự tìm repo root bằng marker `src/models/selection/filter_signal.py`.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    markers = [
        Path('src/models/selection/filter_signal.py'),
        Path('experiments/analysis/analyze_lstm_filter_signal.py'),
    ]
    for candidate in [current, *current.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(f'Cannot find repo root from {current}')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.metric import evaluate
from src.models.selection import filter_signal

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)
print(ROOT)


## Configuration

Default artifact là full run sau khi tách module selector: `portable_lstm_filter_signal_20260509_r06_selector_module`.

Bạn có thể đổi `RUN_NAME` sang:
- `portable_lstm_filter_signal_20260508_r04` để so với pre-refactor plain LSTM.
- `portable_lstm_filter_signal_20260508_r05_signmag` để xem signmag challenger.


In [ ]:
RUN_ROOT = ROOT / 'data/processed/assets/data_info_vn/history/training_runs'
FILTER_ROOT = RUN_ROOT / 'reports/filter_signal'
RUN_NAME = 'portable_lstm_filter_signal_20260509_r06_selector_module'
RUN_DIR = FILTER_ROOT / RUN_NAME

PREDICTIONS_PATH = RUN_DIR / 'filter_predictions.csv.gz'
SUMMARY_PATH = RUN_DIR / 'filter_candidate_summary.csv'
COVERAGE_PATH = RUN_DIR / 'filter_candidate_coverage.csv'
SUMMARY_JSON_PATH = RUN_DIR / 'summary.json'

for required in [PREDICTIONS_PATH, SUMMARY_PATH, COVERAGE_PATH, SUMMARY_JSON_PATH]:
    if not required.exists():
        raise FileNotFoundError(required)

print(RUN_DIR)


## Load Existing Report Artifacts

Các file này do script `experiments/analysis/analyze_lstm_filter_signal.py` tạo ra. Notebook chỉ đọc lại để kiểm tra và thử selector.


In [ ]:
pred = pd.read_csv(PREDICTIONS_PATH, parse_dates=['Date', 'actual_date'])
reported_summary = pd.read_csv(SUMMARY_PATH)
reported_coverage = pd.read_csv(COVERAGE_PATH)
summary_json = json.loads(SUMMARY_JSON_PATH.read_text(encoding='utf-8'))

print(pred.shape)
display(pred.head())
display(reported_summary.sort_values(['split', 'rel_score'], ascending=[True, False]).head(8))
display(reported_summary.query("split == 'val'").sort_values('rel_score', ascending=False).head(8))
print(summary_json.get('selection_params', {}))


## Local Metric Function

Selector module nhận `metric_fn(frame, prediction_column)` để chọn params trên train. Hàm dưới mirror logic chính trong analysis script nhưng giữ gọn để notebook dễ đọc.


In [ ]:
def metric_bundle(frame: pd.DataFrame, prediction_column: str) -> dict[str, float]:
    clean = frame.dropna(subset=[prediction_column, 'actual_aligned']).copy()
    if len(clean) < 10:
        return {
            'n_obs': int(len(clean)),
            'rel_score': float('nan'),
            'directional_accuracy': float('nan'),
            'mean_daily_ic': float('nan'),
            'daily_ic_t': float('nan'),
            'quartile_equity': float('nan'),
            'quartile_hit_rate': float('nan'),
        }

    group_ids = clean['code'].astype(str).to_numpy() if 'code' in clean.columns else None
    scores = evaluate(
        clean[prediction_column].to_numpy(dtype=float),
        clean['actual_aligned'].to_numpy(dtype=float),
        group_ids=group_ids,
    )

    daily_ics: list[float] = []
    long_short: list[float] = []
    for _, day in clean.groupby('actual_date', sort=True):
        day = day.dropna(subset=[prediction_column, 'actual_aligned'])
        if len(day) < 20 or day[prediction_column].nunique() < 3:
            continue
        daily_ics.append(float(day[prediction_column].rank().corr(day['actual_aligned'].rank())))
        rank_pct = day[prediction_column].rank(method='first', pct=True)
        top = day[rank_pct >= 0.8]
        bottom = day[rank_pct <= 0.2]
        if len(top) and len(bottom):
            long_short.append(float(top['actual_aligned'].mean() - bottom['actual_aligned'].mean()))

    ic_arr = np.asarray([value for value in daily_ics if np.isfinite(value)], dtype=float)
    ls_arr = np.asarray([value for value in long_short if np.isfinite(value)], dtype=float)
    ic_std = float(np.std(ic_arr, ddof=1)) if len(ic_arr) > 1 else float('nan')
    return {
        'n_obs': int(len(clean)),
        'rel_score': float(scores['rel_score']),
        'directional_accuracy': float(scores['directional_accuracy']),
        'mean_daily_ic': float(np.mean(ic_arr)) if len(ic_arr) else float('nan'),
        'daily_ic_t': float(np.mean(ic_arr) / (ic_std / np.sqrt(len(ic_arr)))) if len(ic_arr) > 1 and ic_std > 0 else float('nan'),
        'quartile_equity': float(np.prod(1.0 + ls_arr)) if len(ls_arr) else float('nan'),
        'quartile_hit_rate': float(np.mean(ls_arr > 0.0)) if len(ls_arr) else float('nan'),
    }


## Refit Selector From Train Only

Cell này là phần quan trọng nhất: notebook gọi module `src.models.selection.filter_signal`, fit params chỉ từ split `train`, rồi apply cùng params cho toàn bộ frame.


In [ ]:
base_cols = ['code', 'split', 'Date', 'actual_date', 'actual_aligned', 'base_prediction', 'filter_probability']
work = pred[base_cols].copy()

params = filter_signal.fit_filter_signal_selection(work, metric_bundle)
rescored, candidate_columns = filter_signal.apply_filter_signal_selection(work, params)
coverage = pd.DataFrame(filter_signal.candidate_coverage_bundle(rescored, candidate_columns))

print(params)
display(coverage.sort_values(['split', 'coverage']).reset_index(drop=True))


## Compare Notebook Metrics With Script Report

Nếu refactor đúng, `move_top_train_ic_selected` trong notebook phải khớp report CSV của full run.


In [ ]:
rows = []
for split, group in rescored.groupby('split', sort=True):
    for candidate, column in candidate_columns.items():
        row = {'split': split, 'candidate': candidate}
        row.update(metric_bundle(group, column))
        rows.append(row)
notebook_summary = pd.DataFrame(rows)

compare_cols = ['split', 'candidate', 'rel_score', 'directional_accuracy', 'mean_daily_ic', 'daily_ic_t', 'quartile_equity', 'quartile_hit_rate']
comparison = notebook_summary[compare_cols].merge(
    reported_summary[compare_cols],
    on=['split', 'candidate'],
    suffixes=('_notebook', '_reported'),
)
for metric in ['rel_score', 'directional_accuracy', 'mean_daily_ic', 'daily_ic_t', 'quartile_equity', 'quartile_hit_rate']:
    comparison[f'{metric}_diff'] = comparison[f'{metric}_notebook'] - comparison[f'{metric}_reported']

display(comparison.query("split == 'val'").sort_values('rel_score_notebook', ascending=False).head(10))
print('max_abs_rel_score_diff:', comparison['rel_score_diff'].abs().max())


## Inspect Best Candidate

Current best candidate là `move_top_train_ic_selected`.


In [ ]:
BEST = 'move_top_train_ic_selected'
best_col = candidate_columns[BEST]

best_report = notebook_summary.query("candidate == @BEST").copy()
best_coverage = coverage.query("candidate == @BEST").copy()
display(best_report)
display(best_coverage)

selected = rescored[rescored[best_col] != 0].copy()
print('selected rows:', len(selected), 'of', len(rescored))
display(
    selected.groupby('split').agg(
        rows=('code', 'size'),
        names_per_day=('code', lambda s: s.size / selected.loc[s.index, 'actual_date'].nunique()),
        avg_abs_prediction=('base_prediction', lambda s: s.abs().mean()),
        avg_filter_probability=('filter_probability', 'mean'),
    )
)


## Try Custom Coverage Candidates

Dùng cell này để thử coverage grid khác. Lưu ý: vẫn fit trên train, apply sang val.


In [ ]:
CUSTOM_COVERAGE_CANDIDATES = (0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50)

custom_params = filter_signal.fit_filter_signal_selection(
    work,
    metric_bundle,
    coverage_candidates=CUSTOM_COVERAGE_CANDIDATES,
)
custom_scored, custom_candidates = filter_signal.apply_filter_signal_selection(work, custom_params)

custom_rows = []
for split, group in custom_scored.groupby('split', sort=True):
    for candidate, column in custom_candidates.items():
        row = {'split': split, 'candidate': candidate}
        row.update(metric_bundle(group, column))
        custom_rows.append(row)
custom_summary = pd.DataFrame(custom_rows)

print(custom_params)
display(custom_summary.query("split == 'val'").sort_values('rel_score', ascending=False).head(10))


## Quick Risk Read

Risk scaling hiện tại đang bị reject trong report. Cell này giúp xem lại tác động của risk-scaled candidates nếu bạn muốn thử rule mới sau này.


In [ ]:
risk_candidates = [c for c in candidate_columns if 'risk_scaled' in c]
display(
    notebook_summary[
        (notebook_summary['split'] == 'val') & (notebook_summary['candidate'].isin(['base', BEST, *risk_candidates]))
    ].sort_values('rel_score', ascending=False)
)


## Notes / Next Experiments

- Chạy notebook với `RUN_NAME = portable_lstm_filter_signal_20260508_r05_signmag` để xem signmag.
- Thêm rolling-window split nếu muốn kiểm tra stability qua regime.
- Trước khi mở holdout/test, nên thêm turnover và transaction cost read.
